In [ ]:
!pip install scikit-learn scipy pandas --quiet

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist,squareform
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [ ]:
url="https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
df=pd.read_csv(url,header=None, names=["sepal_length","sepal_width","petal_length","petal_width","species"])

In [ ]:
print("Shape:",df.shape)
print("\nFirst 5 rows:")
df.head()

Shape: (150, 5)

First 5 rows:


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [ ]:
X=df[["sepal_length","sepal_width","petal_length","petal_width"]].values

In [ ]:
print("Feature matrix shape:",X.shape)
print("\nFirst 3 rows of X:")
print(X[:3])

Feature matrix shape: (150, 4)

First 3 rows of X:
[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]]


In [ ]:
dist_vector=pdist(X,metric="euclidean")
dist_matrix=squareform(dist_vector)

print("\nDistance matrix shape:",dist_matrix.shape)
print("\nSample distances (first 3x3):")
print(dist_matrix[:3,:3])


Distance matrix shape: (150, 150)

Sample distances (first 3x3):
[[0.         0.53851648 0.50990195]
 [0.53851648 0.         0.3       ]
 [0.50990195 0.3        0.        ]]


In [10]:
sparse_matrix=csr_matrix(dist_matrix)
mst=minimum_spanning_tree(sparse_matrix)

In [11]:
print("Number of edges in MST:",mst.nnz)

Number of edges in MST: 149


In [13]:
print("\nMST (first few edges):")
print("\nMST (first few edges):")
mst_coo=mst.tocoo()
for i in range(5):
  print(f"Node{mst_coo.row[i]} -> Node{mst_coo.col[i]} | Distance: {mst_coo.data[i]:.4f}")



MST (first few edges):

MST (first few edges):
Node0 -> Node4 | Distance: 0.1414
Node0 -> Node17 | Distance: 0.1000
Node0 -> Node27 | Distance: 0.1414
Node0 -> Node39 | Distance: 0.1414
Node1 -> Node9 | Distance: 0.1732


In [16]:
k=3
edges=list(zip(mst_coo.row,mst_coo.col,mst_coo.data))
print(f"Total edges in MST: {len(edges)}")
print("\nTop 5 heaviest edges (candidates for cutting):")
edges_sorted=sorted ( edges , key=lambda e: e[2],reverse=True)
for i in range(5):
  u,v,w=edges_sorted[i]
  print(f"Node {u} -> Node {v} | Distance: {w:.4f}")

Total edges in MST: 149

Top 5 heaviest edges (candidates for cutting):
Node 23 -> Node 98 | Distance: 1.6401
Node 105 -> Node 117 | Distance: 0.8185
Node 84 -> Node 106 | Distance: 0.7348
Node 81 -> Node 93 | Distance: 0.6481
Node 109 -> Node 143 | Distance: 0.6325


In [17]:
removed = set()
for i in range(k-1):
  u,v, _ = edges_sorted[i]
  removed.add((u,v))
  removed.add((v,u))
print(f"n\nRemoved {k-1} edge(s) to create {k} ckusters.")

n
Removed 2 edge(s) to create 3 ckusters.


In [25]:
n_samples=X.shape[0]
adj={i: [] for i in range(n_samples)}
for u,v,w in edges:
  if(u,v) not in removed:
    adj[u].append(v)
    adj[v].append(u)

print("adjacency list for node0:", adj[0])
print("adjacency list for node1:", adj[1])

adjacency list for node0: [np.int32(4), np.int32(17), np.int32(27), np.int32(39)]
adjacency list for node1: [np.int32(9), np.int32(12), np.int32(34), np.int32(37), np.int32(45)]


In [29]:
labels_mst = np.full(n_samples, -1, dtype=int)
cluster_id = 0

for start in range(n_samples):
    if labels_mst[start] != -1:
        continue

    queue = [start]
    labels_mst[start] = cluster_id

    while queue:
        node = queue.pop()
        for neighbour in adj[node]:
            if labels_mst[neighbour] == -1:
                labels_mst[neighbour] = cluster_id
                queue.append(neighbour)

    cluster_id += 1

print("\nMST Cluster labels (first 10):", labels_mst[:10])
print("Unique clusters found:", np.unique(labels_mst))
print("\nCluster sizes:")
for c in np.unique(labels_mst):
    print(f"  Cluster {c}: {np.sum(labels_mst == c)} samples")


MST Cluster labels (first 10): [0 0 0 0 0 0 0 0 0 0]
Unique clusters found: [0 1 2]

Cluster sizes:
  Cluster 0: 50 samples
  Cluster 1: 98 samples
  Cluster 2: 2 samples


In [31]:
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)
print("Original First row:",X[0])
print("Scaled First row:",X_scaled[0].round(4))

kmeans=KMeans(n_clusters=3,n_init=20,random_state=42)
labels_kmeans=kmeans.fit_predict(X_scaled)

print('\nK-means Cluster labels (first 10):',labels_kmeans[:10])
print('Unique clusters found:', np.unique(labels_kmeans))
print('\nCluster sizes:')
for c in np.unique(labels_kmeans):
  print(f"cluster {c}: {np.sum(labels_kmeans ==c)} samples")

Original First row: [5.1 3.5 1.4 0.2]
Scaled First row: [-0.9007  1.0321 -1.3413 -1.313 ]

K-means Cluster labels (first 10): [1 1 1 1 1 1 1 1 1 1]
Unique clusters found: [0 1 2]

Cluster sizes:
cluster 0: 53 samples
cluster 1: 50 samples
cluster 2: 47 samples


In [32]:
sil_mst=silhouette_score(X,labels_mst)
sil_kmeans=silhouette_score(X,labels_kmeans)

print("=" * 40)
print("   COMPARISON REPORT")
print("=" *40)
print(f"MST CLUSTERING SILHOUETTE: {sil_mst:.4f}")
print(f"K-Means   silhouette: {sil_kmeans:.4f}")
print("="*40)

if sil_mst > sil_kmeans:
  print("MST clustering performed better.")
else:
  print("K-Means clustering performed better.")

print("\nNote:")
print("MST is better at capturing non spherical clusters")
print("K-means is better when clusters are round and balanced")

   COMPARISON REPORT
MST CLUSTERING SILHOUETTE: 0.5118
K-Means   silhouette: 0.5059
MST clustering performed better.

Note:
MST is better at capturing non spherical clusters
K-means is better when clusters are round and balanced


In [33]:
mst_df = pd.DataFrame({
    "SampleId": range(n_samples),
    "ClusterLabel": labels_mst
})
mst_df.to_csv("mst_clusters.csv", index=False)
print("mst_clusters.csv saved!")
print(mst_df.head())
kmeans_df = pd.DataFrame({
    "SampleId": range(n_samples),
    "ClusterLabel": labels_kmeans
})
kmeans_df.to_csv("kmeans_clusters.csv", index=False)
print("\nkmeans_clusters.csv saved!")
print(kmeans_df.head())

print("\nAll files saved! Your deliverables are ready.")


mst_clusters.csv saved!
   SampleId  ClusterLabel
0         0             0
1         1             0
2         2             0
3         3             0
4         4             0

kmeans_clusters.csv saved!
   SampleId  ClusterLabel
0         0             1
1         1             1
2         2             1
3         3             1
4         4             1

All files saved! Your deliverables are ready.


In [34]:
from google.colab import files
files.download("mst_clusters.csv")
files.download("kmeans_clusters.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Why MST Can Capture Non-Spherical Clusters




MST-based clustering does not assume any particular shape for clusters, unlike K-Means which assumes clusters are spherical and equally sized. It works by building a graph where every data point is connected to every other point via weighted edges representing distances. The Minimum Spanning Tree retains only the most essential connections, preserving the natural structure of the data. Clusters are formed by removing the (k−1) heaviest edges, which correspond to the largest gaps between groups of points. This allows MST to detect elongated, irregular, or chain-like clusters that K-Means would fail to separate correctly. For example, two interleaving crescent shapes would be impossible for K-Means to separate, but MST would capture them naturally. However, MST is sensitive to outliers and noise — a single outlier point can act as a bridge between two clusters, causing the chaining effect. This was observed in our Iris experiment where Versicolor and Virginica were partially merged into one large cluster of 98 points. Despite this, MST correctly isolated Iris-Setosa perfectly due to the large distance gap separating it from the other two species.